In [1]:
# ============================================================
# Section 1: Imports and Photos Library paths
# ============================================================

import osxphotos

from explorephotoslibrary import *


USE_INVENTORY_CACHE = False


PHOTOS_LIBRARY_PATHS = {
    "backup_20250317": "/Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary",
    "snapshot_20260603": "/Volumes/PRO-G40--20260519/Photos Library-Snapshot--20260603181919/Photos Library.photoslibrary",
    "test": "/Users/huohsien/Pictures/test.photoslibrary"
}

In [2]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

if USE_INVENTORY_CACHE:
    print("=" * 80)
    print("Load inventory cache: backup_20250317")
    print("=" * 80)

    inventory_backup = load_inventory_cache("backup_20250317")

    print()
    print("=" * 80)
    print("Load inventory cache: snapshot_20260603")
    print("=" * 80)

    inventory_snapshot = load_inventory_cache("snapshot_20260603")

else:
    print("=" * 80)
    print("Build inventory: backup_20250317")
    print("=" * 80)

    osx_assets = osxphotos.PhotosDB(
        PHOTOS_LIBRARY_PATHS["backup_20250317"]
    ).photos()

    print("backup osx asset count:", len(osx_assets))

    inventory_backup = build_inventory(osx_assets)

    print()
    print("Backup inventory summary")
    print("------------------------")
    print_inventory_summary(inventory_backup)

    save_inventory_cache(inventory_backup, "backup_20250317")

    print()
    print("=" * 80)
    print("Build inventory: snapshot_20260603")
    print("=" * 80)

    osx_assets_snapshot = osxphotos.PhotosDB(
        PHOTOS_LIBRARY_PATHS["snapshot_20260603"]
    ).photos()

    print("snapshot osx asset count:", len(osx_assets_snapshot))

    inventory_snapshot = build_inventory(osx_assets_snapshot)

    print()
    print("Snapshot inventory summary")
    print("--------------------------")
    print_inventory_summary(inventory_snapshot)

    save_inventory_cache(inventory_snapshot, "snapshot_20260603")

Build inventory: backup_20250317
backup osx asset count: 71599
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000

Backup inventory summary
------------------------
inventory assets: 71599
inventory albums: 5172
inventory folders: 35
movies: 6240
hidden: 0
favorites: 699
descriptions: 727
keywords: 23758
saved inventory cache: data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seconds: 0.82

Build inventory: snapshot_20260603


FileNotFoundError: [Errno dbfile /Volumes/PRO-G40--20260519/Photos Library-Snapshot--20260603181919/Photos Library.photoslibrary does not exist] /Volumes/PRO-G40--20260519/Photos Library-Snapshot--20260603181919/Photos Library.photoslibrary

In [ ]:
hidden_assets = [asset for asset in inventory_backup["assets"] if asset["hidden"] is True]

print("hidden asset count:", len(hidden_assets))
hidden_assets[:5]

In [ ]:
# ============================================================
# Dump all folders
# ============================================================

folders = inventory_backup["folders"]

print("folder count:", len(folders))
print()

for folder_uuid, folder in sorted(
    folders.items(),
    key=lambda item: item[1]["path"]
):
    print("uuid:", folder_uuid)
    print("title:", folder["title"])
    print("path:", folder["path"])
    print()

In [ ]:
# ============================================================
# Find folder by title keyword
# ============================================================

target_folder_title = "4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料）"

matched_folders = find_folders_by_title_keyword(
    inventory_backup,
    target_folder_title,
)

print("matched folder count:", len(matched_folders))
print()

for folder in matched_folders:
    print("folder uuid:", folder["uuid"])
    print("folder title:", folder["title"])
    print("folder path:", folder["path"])
    print()

In [ ]:
# ============================================================
# List albums under matched_folders[0]
# ============================================================

folder = matched_folders[0]

print("Target folder")
print("-------------")
print("folder title:", folder["title"])
print("folder path:", folder["path"])
print("folder uuid:", folder["uuid"])
print()

matched_albums = find_albums_under_folder(
    inventory_backup,
    folder,
)

print("album count:", len(matched_albums))
print()

for album in matched_albums:
    print(album["title"])

In [ ]:
# ============================================================
# Check whether old folder's albums still exist in snapshot
# ============================================================

old_album_titles = [
    album["title"]
    for album in matched_albums
]

found_titles, missing_titles = check_album_titles_exist_in_inventory(
    album_titles=old_album_titles,
    inventory=inventory_snapshot,
)

print("old album count:", len(old_album_titles))
print("found in snapshot:", len(found_titles))
print("missing in snapshot:", len(missing_titles))
print()

print("Missing albums")
print("--------------")
for title in missing_titles:
    print(title)

print()
print("Found albums")
print("------------")
for title in found_titles:
    print(title)

In [ ]:
find_folders_by_title_keyword(inventory_backup,"外送")

In [ ]:
# ============================================================
# Test 1: Find assets with exact HIDE keyword
# ============================================================

hide_keyword_assets = [
    asset
    for asset in inventory_snapshot["assets"]
    if "HIDE" in tuple(asset["keywords"] or ())
]

print("HIDE keyword asset count:", len(hide_keyword_assets))
hide_keyword_assets[:5]

In [ ]:
# ============================================================
# Test 1: Check whether a proposed asset unique ID scheme is unique
# ============================================================

def check_photos_library_asset_unique_id_scheme(
    photos_library_path,
    max_duplicate_groups_to_print=20,
):
    # This function checks whether the proposed asset unique ID scheme is unique
    # inside one Photos Library.
    #
    # Proposed ID scheme:
    #   full original_filename
    #   + date converted to Asia/Taipei
    #   + year removed
    #   + keep month/day/hour/minute/second
    #
    # Example:
    #   IMG_0164.PNG + 11-28 13:18:00

    import osxphotos
    from collections import defaultdict
    from zoneinfo import ZoneInfo

    photosdb = osxphotos.PhotosDB(photos_library_path)
    assets = photosdb.photos()

    taipei_tz = ZoneInfo("Asia/Taipei")

    key_to_assets = defaultdict(list)
    assets_without_key = []

    for asset in assets:
        original_filename = asset.original_filename or asset.filename

        if original_filename is None or asset.date is None:
            assets_without_key.append(asset)
            continue

        # Normalize date/time to Asia/Taipei before removing the year.
        date_taipei = asset.date.astimezone(taipei_tz)

        no_year_datetime = (
            f"{date_taipei.month:02d}-"
            f"{date_taipei.day:02d} "
            f"{date_taipei.hour:02d}:"
            f"{date_taipei.minute:02d}:"
            f"{date_taipei.second:02d}."
            f"{date_taipei.microsecond:06d}"
)

        # Normalize date_added to Asia/Taipei and keep the full timestamp.
        date_added_taipei = asset.date_added.astimezone(taipei_tz)

        date_added_datetime = (
            f"{date_added_taipei.year:04d}-"
            f"{date_added_taipei.month:02d}-"
            f"{date_added_taipei.day:02d} "
            f"{date_added_taipei.hour:02d}:"
            f"{date_added_taipei.minute:02d}:"
            f"{date_added_taipei.second:02d}"
        )

        proposed_unique_id = (
            original_filename,
            no_year_datetime,
            date_added_datetime,
        )

        key_to_assets[proposed_unique_id].append(asset)

    duplicate_groups = {
        key: group
        for key, group in key_to_assets.items()
        if len(group) > 1
    }

    duplicate_asset_count = sum(len(group) for group in duplicate_groups.values())
    is_unique = len(duplicate_groups) == 0 and len(assets_without_key) == 0

    print("Photos Library:")
    print(photos_library_path)
    print("-" * 80)
    print("total asset count:", len(assets))
    print("generated unique ID count:", len(key_to_assets))
    print("assets without proposed unique ID:", len(assets_without_key))
    print("duplicate unique ID group count:", len(duplicate_groups))
    print("duplicate asset count:", duplicate_asset_count)
    print("is proposed unique ID scheme unique:", is_unique)

    if assets_without_key:
        print()
        print("Assets without proposed unique ID")
        print("-" * 80)

        for asset in assets_without_key[:max_duplicate_groups_to_print]:
            print("uuid:", asset.uuid)
            print("filename:", asset.filename)
            print("original_filename:", asset.original_filename)
            print("date:", asset.date)
            print("-" * 80)

    if duplicate_groups:
        print()
        print("Duplicate unique ID groups")
        print("-" * 80)

        for index, (key, group) in enumerate(duplicate_groups.items()):
            if index >= max_duplicate_groups_to_print:
                print("... more duplicate groups not printed")
                break

            print("proposed unique ID:", key)
            print("asset count:", len(group))

            for asset in group:
                print("  uuid:", asset.uuid)
                print("  filename:", asset.filename)
                print("  original_filename:", asset.original_filename)
                print("  date:", asset.date)
                print("  path:", asset.path)

            print("-" * 80)

    return is_unique

In [ ]:
check_photos_library_asset_unique_id_scheme(PHOTOS_LIBRARY_PATHS["snapshot_20260603"])

In [ ]:
check_photos_library_asset_unique_id_scheme(PHOTOS_LIBRARY_PATHS["backup_20250317"])